In [28]:
import json
import pandas as pd
import csv

In [2]:
raw_data = []
with open('final_raw_data_full.json') as d:
  raw_data = json.load(d)

print(len(raw_data))

4603


In [53]:
blacklist_keywords = ['Christmas and Thanksgiving', "Saint Patrick's Day", "New Year's Eve", "Romantic and Valentine's Day", "Halloween", "4th July and Independence Day", "Hall of Fame and must know/try", "Modern", "Classic/vintage", "Contemporary classic", "Party", "Barbeque", "Tennis and Wimbledon", "Breakfast/brunch", "Elevenses/afternoon", "Anytime", "After dinner/digestif", "Summer", "Winter", "Autumn/fall", "Spring", "Layered", "Frappé", "Hot Drinks", "Sugar-free and low-calorie", "Shot cocktails", "Frozen (blended)", "Champagne", "Martini-style", "Long drinks and highballs", "Short and stirred", "Nightcap/sipping", "Spirit-forward", "Aperitivo/aperitif"]
filtered_raw_data = []
for dat in raw_data:
    obj = {}
    if 'ingridients' in dat:
        obj['ingridients'] = dat['ingridients']
    if 'keywords' in dat:
        keys = []
        for k in dat['keywords']:
            if k not in blacklist_keywords:
                if '(e.g.' in k:
                    k = k.split('(e.g')[0].strip()
                keys.append(k)
        obj['keywords'] = keys
    if 'strength_rating' in dat:
        obj['strength_rating'] = int(dat['strength_rating'].replace('Strength', '').replace('/10', ''))
    if 'taste_rating' in dat:
        obj['taste_rating'] = int(dat['taste_rating'].replace('Sweet to sour ', '').replace('/10', ''))
    if 'rating_data' in dat:
        obj['rating'] = dat['rating_data']['ratingValue']
    filtered_raw_data.append(obj)

print(len(filtered_raw_data))

4603


In [54]:
keywordMap = {}
for dat in filtered_raw_data:
    for word in dat['keywords']:
        if word not in keywordMap:
            keywordMap[word] = 1
        else:
            keywordMap[word] += 1

print(keywordMap)
len(keywordMap)

{'Fruity': 660, "Pimm's Cup": 1, 'Savoury': 92, 'Red Snapper': 1, 'Tiki/tropical': 125, 'Bittersweet': 486, 'Pre-batched': 2, 'Sake-Tini': 2, 'Ferrobaldi Cocktail': 1, 'Paper Airplane': 1, 'Herbal': 443, 'Citrusy': 500, 'Sours (citrus)': 703, 'whisky sour': 1, 'Punch': 4, 'coffee martini': 1, 'espresso cocktail': 1, 'expresso martini': 1, 'Tommys Margarita': 1, 'Spicy': 151, 'old-fashioned': 1, 'oldfashioned': 1, 'Old Fashioned Whiskey Cocktail': 1, 'Creamy': 118, 'Ramos Fizz': 1, 'New Orleans Fizz': 1, 'One And Only One': 1, 'Frozen Daiquiri': 1, 'Daiquiri Frappé': 1, 'Fifty Fifty Margarita': 1, 'Champagne Mojito': 1, "Gen'tonique": 1, "Maison Premiere's Mai Tai": 1, 'G&T': 1, 'G and T': 1, 'Gin and Tonic': 1, 'G & T': 1, 'Sweet manhattan': 1, 'M&M': 1, 'Montenegro & Mezcal': 1, 'Montenegro and mezcal': 1, "Manager's Meeting": 1, 'Managers Meeting': 1, 'Monte y Mezcal': 1, 'M and M': 1, 'Dessert cocktails': 148, 'Rum Maple Old Fashioned': 1, 'Macunaima': 1, 'Canchanchara': 1, 'Canchnc

835

In [ ]:
df = pd.DataFrame.from_dict(keywordMap, orient='index', columns=['count'])
df = df.sort_values(by='count', ascending=False)
df = df[df['count'] > 10]
whitelist_keywords = df.index.tolist()

In [56]:
count = 0
for dat in filtered_raw_data:
    keys = [k.lower() for k in dat['keywords'] if k in whitelist_keywords]
    dat['keywords'] = keys
    if len(keys):
        count += 1

print(len(filtered_raw_data), count)

data_with_keywords = [dat for dat in filtered_raw_data if len(dat['keywords'])]
print(len(data_with_keywords))
print(data_with_keywords)

4603 2700
2700
[{'ingridients': ['25 ml Gin', '25 ml Orange Curaçao', '25 ml Rosso/sweet vermouth', '2 dash Aromatic bitters', '50 ml Ginger ale', '50 ml Cola'], 'keywords': ['fruity'], 'strength_rating': 6, 'taste_rating': 6, 'rating': 5}, {'ingridients': ['7 fresh Mint leaves', '45 ml Cognac (brandy)', '15 ml Rye whiskey 50% abv', '7.5 ml Brown sugar syrup', '2 dash Orange bitters'], 'keywords': ['savoury'], 'strength_rating': 7, 'taste_rating': 7, 'rating': 5}, {'ingridients': ['25 ml Light gold rum 1-3yo', '25 ml Orange Curaçao', '25 ml Rosso/sweet vermouth', '2 dash Aromatic bitters', '50 ml Ginger ale', '50 ml Cola'], 'keywords': ['fruity', 'tiki/tropical'], 'strength_rating': 6, 'taste_rating': 6, 'rating': 5}, {'ingridients': ['45 ml Gin', '30 ml Ferro China', '30 ml Rosso/sweet vermouth', '4 drop Mint bitters'], 'keywords': ['bittersweet'], 'strength_rating': 7, 'taste_rating': 7, 'rating': 5}, {'ingridients': ['25 ml Rye whiskey 50% abv', '25 ml Orange Curaçao', '25 ml Rosso/

In [57]:
def extractData(ingridient, keyword):
    if keyword in ingridient:
        splitted = ingridient.split(keyword)
        return [splitted[1].strip().lower(), float(splitted[0].strip()), keyword.strip()]

others = []
for dat in data_with_keywords:
    ingridients = []
    for ingridient in dat['ingridients']:
        if ' ml ' in ingridient:
            ingridients.append(extractData(ingridient, ' ml '))
        elif ' dash ' in ingridient:
            ingridients.append(extractData(ingridient, ' dash '))
        elif ' drop ' in ingridient:
            ingridients.append(extractData(ingridient, ' drop '))
        elif '1&frasl;' in ingridient:
            splitted = ingridient.split('1&frasl;')
            fraction_part = splitted[0].strip()
            rest_part = splitted[1].strip()
            if ' ' in rest_part:
                denom, name = rest_part.split(' ', 1)
                value = 1 / float(denom)
                ingridients.append([name.strip().lower(), value, 'N/A'])
            else:
                others.append(ingridient)
        elif '3&frasl;4' in ingridient:
            splitted = ingridient.split('3&frasl;4')
            fraction_part = splitted[0].strip()
            rest_part = splitted[1].strip()
            ingridients.append([rest_part.lower(), 3/4, 'N/A'])
        elif ' fresh ' in ingridient:
            ingridients.append(extractData(ingridient, ' fresh '))
        elif ' barspoon ' in ingridient:
            ingridients.append(extractData(ingridient, ' barspoon '))
        elif ' pinch ' in ingridient:
            ingridients.append(extractData(ingridient, ' pinch '))
        elif ' sprig ' in ingridient:
            ingridients.append(extractData(ingridient, ' sprig '))
        elif ' swath ' in ingridient:
            ingridients.append(extractData(ingridient, ' swath '))
        elif ' dried ' in ingridient:
            ingridients.append(extractData(ingridient, ' dried '))
        elif ' slice ' in ingridient:
            ingridients.append(extractData(ingridient, ' slice '))
        elif ' grind ' in ingridient:
            ingridients.append(extractData(ingridient, ' grind '))
        elif ' wedge ' in ingridient:
            ingridients.append(extractData(ingridient, ' wedge '))
        elif ' whole ' in ingridient:
            ingridients.append(extractData(ingridient, ' whole '))
        elif ' twist ' in ingridient:
            ingridients.append(extractData(ingridient, ' twist '))
        elif ' ring ' in ingridient:
            ingridients.append(extractData(ingridient, ' ring '))
        elif ' inch ' in ingridient:
            ingridients.append(extractData(ingridient, ' inch '))
        elif ' cube ' in ingridient:
            ingridients.append(extractData(ingridient, ' cube '))
        elif ' pea ' in ingridient:
            ingridients.append(extractData(ingridient, ' pea '))
        elif ' disc ' in ingridient:
            ingridients.append(extractData(ingridient, ' disc '))
        elif ' scoop ' in ingridient:
            ingridients.append(extractData(ingridient, ' scoop '))
        elif ' unit ' in ingridient:
            ingridients.append(extractData(ingridient, ' unit '))
        elif ' cupful ' in ingridient:
            ingridients.append(extractData(ingridient, ' cupful '))
        elif 'Top up with' in ingridient:
            ingridients.append([ingridient.replace('Top up with', '').strip().lower(), 0, 'N/A'])
        else:
            ingridients.append([ingridient.strip().lower(), 0, 'N/A'])
    dat['ingridients_processed'] = ingridients
print(data_with_keywords)

[{'ingridients': ['25 ml Gin', '25 ml Orange Curaçao', '25 ml Rosso/sweet vermouth', '2 dash Aromatic bitters', '50 ml Ginger ale', '50 ml Cola'], 'keywords': ['fruity'], 'strength_rating': 6, 'taste_rating': 6, 'rating': 5, 'ingridients_processed': [['gin', 25.0, 'ml'], ['orange curaçao', 25.0, 'ml'], ['rosso/sweet vermouth', 25.0, 'ml'], ['aromatic bitters', 2.0, 'dash'], ['ginger ale', 50.0, 'ml'], ['cola', 50.0, 'ml']]}, {'ingridients': ['7 fresh Mint leaves', '45 ml Cognac (brandy)', '15 ml Rye whiskey 50% abv', '7.5 ml Brown sugar syrup', '2 dash Orange bitters'], 'keywords': ['savoury'], 'strength_rating': 7, 'taste_rating': 7, 'rating': 5, 'ingridients_processed': [['mint leaves', 7.0, 'fresh'], ['cognac (brandy)', 45.0, 'ml'], ['rye whiskey 50% abv', 15.0, 'ml'], ['brown sugar syrup', 7.5, 'ml'], ['orange bitters', 2.0, 'dash']]}, {'ingridients': ['25 ml Light gold rum 1-3yo', '25 ml Orange Curaçao', '25 ml Rosso/sweet vermouth', '2 dash Aromatic bitters', '50 ml Ginger ale', 

In [58]:
ingridient_list = {}
for dat in data_with_keywords:
    ingridients = dat['ingridients_processed']
    for ingridient in ingridients:
        if ingridient[0] in ingridient_list:
            ingridient_list[ingridient[0]] += 1
        else:
            ingridient_list[ingridient[0]] = 1

# print(len(ingridient_list), ingridient_list)
less_than_10 = []
for ingridient, count in ingridient_list.items():
    if count < 10:
        less_than_10.append(ingridient)
print(len(less_than_10), less_than_10)

385 ['mint bitters', 'lavender bitters', 'amargo bitters', 'green grapes (seedless)', 'red grapefruit juice', 'vanilla extract', 'egg white', 'hot filter coffee', 'pineapple rum', 'aged agricole rhum', 'becherovka liqueur', 'génépi liqueur', 'lemon bitters', 'pineapple liqueur', 'clementi antico', 'framboise eau-de-vie', 'celery gin', 'celery saccharum', 'armagnac brandy', 'black walnut bitters', 'fresh banana', 'palo cortado sherry', 'clove', 'parfait amour liqueur', 'rosé/rosato vermouth', 'mezcal liqueur', 'vanilla sugar', 'pineapple (fresh)', 'sugar cane syrup', 'élixir végétal', 'medium dry cider', 'sour cherry gin', "abbott's bitters", 'red grapes (seedless)', 'overproof brandy', 'aged grappa', 'apricot eau-de-vie', 'tiki bitters', 'corn/maize liqueur', 'mexican corn whisky', 'sauternes wine', 'robust rouge quinquina', 'japanese whisky', 'ring yellow bell pepper', 'tomato juice', 'celery salt', 'worcestershire sauce', 'overproof gin', 'chai tea (cold)', 'bitter honey aperitif', '

In [59]:
MAPPING = {
    'bitter_herbal_floral': [
        'mint bitters', 'lavender bitters', 'ginseng bitters', 
        'coriander bitters', 'dandelion and burdock bitters', 'tonka bitters'
    ],
    'bitter_fruit_citrus': [
        'lemon bitters', 'lime bitters', 'peach bitters', 'plum bitters',
        'rhubarb bitters', 'cherry bitters', 'cranberry bitters', 'maple bitters'
    ],
    'bitter_spice_wood': [
        "abbott's bitters", 'orinoco bitters', 'cardamom bitters',
        'tobacco bitters', 'black walnut bitters', 'pimento bitters',
        'barrel-aged bitters'
    ],
    'bitter_chocolate_coffee': ['cocoa bitters', 'jerry thomas bitters'],
    'bitter_generic': [
        'amargo bitters', 'tiki bitters', 'bitter honey aperitif',
        'burlesque bitters', 'olive bitters'
    ],
    'fruit_liqueurs': [
        'pineapple liqueur',       # Tropical fruit
        'sour apple liqueur',      # Tart fruit
        'strawberry liqueur',      # Berry fruit
        'pomegranate liqueur',     # Tart berry fruit
        'lychee liqueur',          # Tropical fruit
        'blood orange liqueur',    # Citrus fruit
        'fig liqueur',             # Sweet fruit
        'quince liqueur',          # Tart fruit
        'kumquat liqueur',         # Citrus fruit
        'blueberry liqueur',       # Berry fruit
        'lapponia cloudberry liqueur', # Berry fruit
        'green banana liqueur',    # Tropical fruit
        'pear liqueur',            # Orchard fruit
        'spiced pear liqueur'      # Fruit + spice (primarily fruit)
    ],
    'citrus_liqueurs': [
        'créole shrubb liqueur',   # Orange-based
        'yuzushu liqueur',         # Japanese citrus
        'china-china liqueur',     # Orange/bitter orange
        'agave sec liqueur'        # Agave + citrus notes
    ],
    'herbal_floral_liqueurs': [
        'génépi liqueur',          # Alpine herb (artemisia)
        'lavender liqueur',        # Floral
        'thyme liqueur',           # Herb
        'sorrel liqueur',          # Herb/tart
        'vetiver gris liqueur',    # Grassy/herbal
        'rhubarb liqueur',         # Tart botanical (not sweet fruit)
        'chamomile/camomile liqueur', # Floral/herbal
        'acqua bianca liqueur',    # Herbal Italian
        'ginseng liqueur'          # Herbal/earthy
    ],
    'nut_spice_liqueurs': [
        'orgeat almond liqueur',   # Almond (nut)
        'walnut liqueur',          # Nut
        'mastiha liqueur',         # Resin/spice
        'corn/maize liqueur',      # Grain/spice
        'krupnik liqueur',         # Honey/spice
        'pistachio cream liqueur'  # Nut + cream
    ],
    'bitter_amaro_liqueurs': [
        'becherovka liqueur',      # Herbal bitter
        'mezcal liqueur',          # Smoky/bitter
        'amaro liqueur',           # Generic bitter
        'centerbe liqueur',        # "Hundred herbs" - bitter herbal
        'izarra jaune liqueur',    # Basque herbal bitter
        'chinotto liqueur',        # Bitter citrus
        'bitter almond liqueur',   # Bitter nut (not sweet almond)
        'pacharan liqueur'         # Basque sloe/bitter
    ],
    'anise_liqueurs': [
        'aniseed liqueur',         # Anise
        'anisette liqueur',        # Anise
        'pastis',                  # Anise (though not in list, related)
        'ouzo'                     # Anise (though not in list, related)
    ],
    'chocolate_caramel_liqueurs': [
        'parfait amour liqueur',   # Vanilla/citrus/floral (but often chocolatey)
        'vanilla liqueur',         # Sweet vanilla
        'caramel liqueur',         # Sweet caramel
        'chocolate liqueur',       # Chocolate
        'chocolate orange liqueur',# Chocolate + citrus
        'butterscotch liqueur',    # Butterscotch
        'goldwasser liqueur',      # Spiced citrus with gold flakes
        'tuaca liqueur',           # Vanilla/citrus
        'southern liqueur'         # Often peach/vanilla (Southern Comfort style)
    ],
    'savory_unique_liqueurs': [
        'tomato liqueur',          # Savory vegetable
        'aloe liqueur'             # Herbal/slightly bitter
    ],
    'flavored_rum': ['pineapple rum', 'banana rum', 'cacao spiced rum', 'vanilla rum'],
    'specialty_rum': ['blended light 3-5yo rum', 'overproof aged light rum', '100% pot still rum'],

    # Group 1: Light/Delicate
    'light_delicate_whisky': ['canadian whisky', 'japanese whisky'],

    # Group 2: Unique Grain
    'unique_grain_whisky': ['mexican corn whisky'],

    # Group 3: Scotch Malt Character  
    'scotch_malt_whisky': ['highland malt whisky', 'vatted malt whisky'],

    'infused_flavor_gin': [
        'celery gin', 'sour cherry gin', 'sage gin', 
        'grapefruit gin', 'honey and mint gin'
    ],
    'strength_aged_gin': [
        'overproof gin', 'navy strength gin', 'oak aged gin'
    ],
    'red_wine': ['claret wine', 'pinot noir red wine', 'rioja wine', 'shiraz red wine'],
    'white_wine': ['chardonnay wine', 'viognier white wine', 'soave wine'],
    'sweet_wine': ['sauternes wine', 'moscato d\'asti wine', 'ginger wine'],
    'cider': ['medium dry cider', 'dry cider'],
    'grape_brandy': [
        'armagnac brandy',    # French grape brandy
        'spanish brandy',     # Spanish grape brandy  
        'italian brandy',     # Italian grape brandy
        'overproof brandy'    # High-proof grape brandy (typically)
    ],
    'fruit_brandy': [
        'peach brandy',           # Peach-based
        'apple/cider eau-de-vie'  # Apple-based
    ],
    'citrus_juice': [
        'red grapefruit juice',  # Bitter citrus
        'mandarin juice',        # Sweet citrus
        'citrus juice'           # Generic citrus
    ],
    'sweet_fruit_juice': [
        'red grape juice',
        'mango juice',           # Tropical sweet
        'cherry juice',          # Berry sweet/tart
        'watermelon juice',      # Melon sweet
        'lychee juice drink',    # Tropical sweet
        'peach juice',           # Stone fruit sweet
        'sparkling apple juice'  # Apple sweet + fizz
    ],
    'vegetable_savory_juice': [
        'verjuice',
        'tomato juice',          # Savory (Bloody Mary)
        'beetroot juice',        # Earthy, sweet-savory
        'celery juice',          # Herbal, vegetal
        'carrot juice',          # Sweet-vegetal
        'red bell pepper juice'  # Sweet-vegetal
    ],
    'fruit_syrup': [
        'spicy mango syrup',        # Mango (tropical fruit)
        'lychee/litchi syrup',      # Lychee (tropical fruit)
        'groseille syrup',          # Red currant (berry fruit)
        'strawberry sugar syrup',   # Strawberry (berry fruit)
        'kiwi sugar syrup',         # Kiwi (tropical fruit)
        'apple sugar syrup',        # Apple (orchard fruit)
        'rhubarb syrup',            # Rhubarb (tart fruit)
        'cherry syrup',             # Cherry (stone fruit)
        'watermelon syrup',         # Watermelon (melon fruit)
        'pink grapefruit syrup',    # Grapefruit (citrus fruit)
        'spiced red berries syrup', # Mixed berries (berry fruit)
        'blackcurrant syrup',       # Blackcurrant (berry fruit)
        'peach syrup',              # Peach (stone fruit)
    ],
    'herbal_floral_syrup': [
        'peppermint syrup',
        'celery saccharum',
        'lavender sugar syrup',     # Lavender (floral)
        'rose sugar syrup',         # Rose (floral)
        'cherry blossom syrup',     # Cherry blossom (floral)
        'cucumber syrup',           # Cucumber (vegetal/herbal)
        'basil syrup',              # Basil (herbal)
    ],
    'spice_syrup': [
        'homemade ginger syrup',    # Ginger (spice)
        'pumpkin spice syrup',      # Pumpkin spice blend
        'gingerbread sugar syrup',  # Ginger + spices
        'winter spice syrup',       # Seasonal spice blend
        'cardamom sugar syrup',     # Cardamom (spice)
    ],
    'nut_caramel_syrup': [
        'pistachio syrup',          # Pistachio (nut)
        'maraschino syrup',         # Marasca cherry (nutty notes)
        'caramel syrup',            # Caramel (sweet/burnt sugar)
        'hazelnut syrup',           # Hazelnut (nut)
        'coconut syrup',            # Coconut (tropical nut)
        'macadamia nut syrup',      # Macadamia (nut)
        'salted caramel syrup',     # Caramel + salt
        'marshmallow syrup',        # Vanilla/sweet (like caramel)
    ],
    'simple_syrup': [
        'sugar cane syrup',         # Cane sugar
        'gomme syrup',              # Gum arabic syrup
        'sugar syrup 1:1',          # Basic simple syrup
        'honey syrup (2:1)',        # Honey-based simple
    ],
    'fresh_fruit': [
        'green grapes (seedless)', 'red grapes (seedless)', 'fresh banana',
        'pineapple (fresh)', 'blueberries (fresh)', 'blackberries',
        'cherry tomato (fresh)', 'fresh kiwi fruit (fresh)', 'kiwi fruit (fresh)',
        'slice pineapple (fresh)', 'redcurrants', 'passion fruit purée',
        'watermelon (fresh)', 'orange (fresh)', 'figs',
        'cranberries (fresh)', 'red bell pepper', 'ruby red grapefruit',
        'fresh strawberries (fresh)', 'raisins', 'cherries (destoned)',
        'peach purée', 'mango purée', 'raspberry puree'
    ],
    'fresh_herbs_spices': [
        'clove', 'red chili pepper', 'coriander/cilantro', 'freshly grated nutmeg',
        'green cardamom pods', 'curry leaf', 'oregano (fresh)', 'sage leaves',
        'fresh rosemary sprig', 'fresh thyme', 'fresh dill', 'lemongrass stem',
        'fresh ginger', 'orange zest/swath', 'lime zest (peel)', 'fresh lemon peel',
        'fresh orange peel', 'orange peel', 'inch rosemary sprig', 'rosemary sprig',
        'thai basil leaves', 'mint (fresh) sprigs', 'slice orange (fresh)',
        'grapefruit peel/zest', 'lemon peel'
    ],
    'egg_dairy': [
        'egg white', 'egg (white and yolk)', 'egg yolk', 'fresh egg yolk',
        'vanilla ice cream', 'coconut sorbet', 'raspberry sorbet', 'coconut cream',
        'coconut milk', 'yoghurt (natural)', 'mascarpone cheese', 'evaporated milk',
        'oat milk'
    ],
    'coffee_tea': [
        'hot filter coffee', 'chai tea (cold)', 'cold breakfast tea',
        'cafetière coffee', 'cold black tea', 'cold brew coffee',
        'cold lapsang souchong', 'berry fruit tea (cold)', 'banana tea'
    ],
    'sweeteners': [
        'vanilla extract', 'vanilla sugar', 'orange marmalade', 'strawberry jam',
        'raspberry jam', 'blackcurrant jam (preserve)', 'powdered sugar',
        'honey', 'honey water (1:1)', 'brown sugar', 'sugar cube',
        'pomegranate molasses', 'orange cream citrate'
    ],
    'salt_acid_savory': [
        'celery salt', 'worcestershire sauce', 'white wine vinegar',
        'balsamic vinegar', 'white balsamic', 'olive brine', 'caper brine',
        'salt', 'pinch salt', 'saline solution 10:1',
        'beef bouillon', 'white pepper (ground)', 'dijon mustard', 'wasabi paste',
        'pumpkin puree'
    ],
    'oils_vinegars': [
        'olive oil', 'sesame oil', 'élixir végétal'  # Herbal oil
    ],
    'beer_sparkling': [
        'india pale ale (ipa) beer', 'stout beer', 'dunkel/black lager',
        'pilsner lager', 'sparkling water', 'boiling water', 'soda from siphon',
        'coconut water', 'birch water', 'kombucha', 'celery soda',
        'tonic water (smoked)', 'mediterranean tonic', 'bitter lemon tonic',
        'aromatic tonic water', 'red bitter soda', 'cherry soda', 'lemonade',
        'pineapple soda', 'mandarin soda', 'fig leaf soda', 'banana tea soda',
        'mango soda', 'coconut-pineapple soda', 'white peach and jasmine soda',
        'elderberry and hibiscus soda', 'grape and apricot soda', 'strawberry soda',
        'olive lemonade'
    ],
    'wines_fortified': [
        'palo cortado sherry', 'rosé/rosato vermouth', 'rubino/rosso/rojo vermouth',
        'white port', 'ruby port', 'marsala semisecco (medium)', 'blossom vermouth',
        'pommeau du normandie', 'orancio vino aperitif', 'lillet rouge',
        'rosé champagne', 'cava', 'pinot grigio', 'coffee vermouth'
    ],
    'spirits_liqueurs': [
        'aged agricole rhum', 'clementi antico', 'framboise eau-de-vie',
        'aged grappa', 'apricot eau-de-vie', 'robust rouge quinquina',
        'akvavit / aquavit', 'chili tincture', 'blanche absinthe',
        'amaro (e.g. braulio)', 'mint fernet amaro', 'chinato (barolo/barbera/grignolino)',
        'tsipouro', 'pineau des charentes', 'bourbon bonded-strength',
        'german amaro', 'angostura di amaro', 'amer picon', 'vermouth amaro (car.)',
        'vermouth amaro (coc.)', 'amaro (e.g. nardini)', 'amaro (e.g. ciociaro)',
        'underberg', 'chinotto', 'herboris amaro centerbe', 'crème de noyau',
        'amaro formidabile', 'sambuca', 'batavia arrack', 'spirit',
        'greek spirit', 'peach and orange spirit', 'grapefruit rose spirit',
        'umeshu plum sake', 'rose water', 'peach aperitif'
    ],
    'infused_spirits': [
        'mango vodka', 'pineapple and chili vodka', 'pear vodka',
        'habenero bourbon', 'grapefruit vodka', 'rhubarb flavoured vodka',
        'orange vodka', 'lime vodka', 'cachaça (aged 3+ years)',
        'cardamom and ginger vodka', 'raspberry vodka', 'strawberry flavoured vodka',
        'watermelon flavoured vodka', 'pepper vodka', 'passion fruit vodka',
        'mixed berry flavoured vodka', 'american single malt', 'island single malt',
        'pot still whiskey'
    ],
    'misc_unique': [
        'ring yellow bell pepper', 'green food colour gel', 'cocktail foamer',
        'jalapeño tequila', 'maraschino cherry'
    ]
}

In [60]:
GROUPING_MAP = {}

for key, value in MAPPING.items():
    for v in value:
        GROUPING_MAP[v] = key

print(GROUPING_MAP)

{'mint bitters': 'bitter_herbal_floral', 'lavender bitters': 'bitter_herbal_floral', 'ginseng bitters': 'bitter_herbal_floral', 'coriander bitters': 'bitter_herbal_floral', 'dandelion and burdock bitters': 'bitter_herbal_floral', 'tonka bitters': 'bitter_herbal_floral', 'lemon bitters': 'bitter_fruit_citrus', 'lime bitters': 'bitter_fruit_citrus', 'peach bitters': 'bitter_fruit_citrus', 'plum bitters': 'bitter_fruit_citrus', 'rhubarb bitters': 'bitter_fruit_citrus', 'cherry bitters': 'bitter_fruit_citrus', 'cranberry bitters': 'bitter_fruit_citrus', 'maple bitters': 'bitter_fruit_citrus', "abbott's bitters": 'bitter_spice_wood', 'orinoco bitters': 'bitter_spice_wood', 'cardamom bitters': 'bitter_spice_wood', 'tobacco bitters': 'bitter_spice_wood', 'black walnut bitters': 'bitter_spice_wood', 'pimento bitters': 'bitter_spice_wood', 'barrel-aged bitters': 'bitter_spice_wood', 'cocoa bitters': 'bitter_chocolate_coffee', 'jerry thomas bitters': 'bitter_chocolate_coffee', 'amargo bitters': 

In [61]:
ingridient_list = {}
for dat in data_with_keywords:
    ingridients = dat['ingridients_processed']
    ingridient_final = []
    for ingridient in ingridients:
        print(ingridient)
        ing_name = ingridient[0]
        if ingridient[0] in GROUPING_MAP:
            ing_name = GROUPING_MAP[ingridient[0]]
        ingridient_final.append([ing_name, ingridient[1], ingridient[2]])
        if ingridient[0] in ingridient_list:
            ingridient_list[ing_name] += 1
        else:
            ingridient_list[ing_name] = 1
    dat['ingridients_final'] = ingridient_final

# print(len(ingridient_list), ingridient_list)
print(len(data_with_keywords), data_with_keywords[0])

['gin', 25.0, 'ml']
['orange curaçao', 25.0, 'ml']
['rosso/sweet vermouth', 25.0, 'ml']
['aromatic bitters', 2.0, 'dash']
['ginger ale', 50.0, 'ml']
['cola', 50.0, 'ml']
['mint leaves', 7.0, 'fresh']
['cognac (brandy)', 45.0, 'ml']
['rye whiskey 50% abv', 15.0, 'ml']
['brown sugar syrup', 7.5, 'ml']
['orange bitters', 2.0, 'dash']
['light gold rum 1-3yo', 25.0, 'ml']
['orange curaçao', 25.0, 'ml']
['rosso/sweet vermouth', 25.0, 'ml']
['aromatic bitters', 2.0, 'dash']
['ginger ale', 50.0, 'ml']
['cola', 50.0, 'ml']
['gin', 45.0, 'ml']
['ferro china', 30.0, 'ml']
['rosso/sweet vermouth', 30.0, 'ml']
['mint bitters', 4.0, 'drop']
['rye whiskey 50% abv', 25.0, 'ml']
['orange curaçao', 25.0, 'ml']
['rosso/sweet vermouth', 25.0, 'ml']
['aromatic bitters', 2.0, 'dash']
['ginger ale', 50.0, 'ml']
['cola', 50.0, 'ml']
['calvados apple brandy', 25.0, 'ml']
['orange curaçao', 25.0, 'ml']
['rosso/sweet vermouth', 25.0, 'ml']
['aromatic bitters', 2.0, 'dash']
['ginger ale', 50.0, 'ml']
['cola', 50.

In [62]:
final_data = []

for dat in data_with_keywords:
    final_data.append({
        'ingridients': dat['ingridients_final'],
        'keywords': dat['keywords'],
        'strength_rating': dat['strength_rating'],
        'taste_rating': dat['taste_rating'],
        'rating': dat['rating']
    })

print(len(final_data), final_data)

2700 [{'ingridients': [['gin', 25.0, 'ml'], ['orange curaçao', 25.0, 'ml'], ['rosso/sweet vermouth', 25.0, 'ml'], ['aromatic bitters', 2.0, 'dash'], ['ginger ale', 50.0, 'ml'], ['cola', 50.0, 'ml']], 'keywords': ['fruity'], 'strength_rating': 6, 'taste_rating': 6, 'rating': 5}, {'ingridients': [['mint leaves', 7.0, 'fresh'], ['cognac (brandy)', 45.0, 'ml'], ['rye whiskey 50% abv', 15.0, 'ml'], ['brown sugar syrup', 7.5, 'ml'], ['orange bitters', 2.0, 'dash']], 'keywords': ['savoury'], 'strength_rating': 7, 'taste_rating': 7, 'rating': 5}, {'ingridients': [['light gold rum 1-3yo', 25.0, 'ml'], ['orange curaçao', 25.0, 'ml'], ['rosso/sweet vermouth', 25.0, 'ml'], ['aromatic bitters', 2.0, 'dash'], ['ginger ale', 50.0, 'ml'], ['cola', 50.0, 'ml']], 'keywords': ['fruity', 'tiki/tropical'], 'strength_rating': 6, 'taste_rating': 6, 'rating': 5}, {'ingridients': [['gin', 45.0, 'ml'], ['ferro china', 30.0, 'ml'], ['rosso/sweet vermouth', 30.0, 'ml'], ['bitter_herbal_floral', 4.0, 'drop']], 'ke

In [63]:
json.dump(final_data, open('final_processed_data.json', 'w'), indent=2)

In [4]:
data = []
with open('final_processed_data.json') as d:
  data = json.load(d)

print(len(data))



2700


In [19]:
keywordMap = {
    'bittersweet': 'bittersweet',
    'citrusy': 'citrus',
    'sours (citrus)': 'citrus',
    'creamy': 'creamy',
    'dessert cocktails': 'sweet',
    'floral': 'floral',
    'fruitini': 'fruity',
    'fruity': 'fruity',
    'herbal': 'herbal',
    'nutty': 'nutty',
    'savoury': 'savoury',
    'spicy': 'spicy',
    'tiki/tropical': 'fruity'
}

In [29]:
ing_unit_map = {}
for dat in data:
    total = 0
    processed_ingridients = []
    processed_keywords = set()
    for keyword in dat['keywords']:
        if keyword in keywordMap:
            processed_keywords.add(keywordMap[keyword])
    for ingridient in dat['ingridients']:
        total += ingridient[1]
        if ingridient[0] not in ing_unit_map:
            ing_unit_map[ingridient[0]] = ingridient[2]
    for ingridient in dat['ingridients']:
        ing_name = ingridient[0].strip().lower().replace(' ', '_')
        ing_pct = (ingridient[1] / total) * 100
        processed_ingridients.append([ing_name, ing_pct])
    dat['pct_ingridients'] = processed_ingridients
    dat['processed_keywords'] = list(processed_keywords)
    # print(processed_ingridients)
print(ing_unit_map)
print(len(ing_unit_map))

{'gin': 'ml', 'orange curaçao': 'ml', 'rosso/sweet vermouth': 'ml', 'aromatic bitters': 'dash', 'ginger ale': 'ml', 'cola': 'ml', 'mint leaves': 'fresh', 'cognac (brandy)': 'ml', 'rye whiskey 50% abv': 'ml', 'brown sugar syrup': 'ml', 'orange bitters': 'dash', 'light gold rum 1-3yo': 'ml', 'ferro china': 'ml', 'bitter_herbal_floral': 'drop', 'calvados apple brandy': 'ml', 'scotch whisky': 'ml', 'mezcal': 'ml', 'pisco': 'ml', 'bitter_generic': 'dash', 'vodka': 'ml', 'sake': 'ml', 'dry vermouth': 'ml', 'saline solution 4:1': 'drop', 'bourbon whiskey': 'ml', 'amaro (e.g.  nonino)': 'ml', 'orange-red aperitivo': 'ml', 'lemon juice': 'ml', 'bénédictine d.o.m.': 'ml', 'creole bitters': 'dash', 'sugar syrup (2:1)': 'ml', 'egg white (pasteurised)': 'ml', 'fresh_fruit': 'fresh', 'red bitter liqueur': 'ml', 'lime juice': 'ml', 'soda (club soda) water': 'ml', 'cherry brandy': 'ml', 'triple sec': 'ml', 'pineapple juice': 'ml', 'grenadine syrup': 'ml', 'reposado tequila': 'ml', 'agave syrup': 'ml',

In [30]:
json.dump(data, open('final_data.json', 'w'), indent=2)

In [32]:
all_ingredients = sorted(set(
    ing_name for recipe in data 
    for ing_name, _ in recipe["pct_ingridients"]
))

all_keywords = sorted(set(
    keyword for recipe in data
    for keyword in recipe["processed_keywords"]
))

print(len(all_ingredients))
print(len(all_keywords), all_keywords)

header = ['recipe_id']

for ing in all_ingredients:
    header.append(f'{ing}_pct')
for key in all_keywords:
    header.append(key)
header.extend(['strength_rating', 'taste_rating', 'rating'])

with open("cocktail_dataset.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=header)
    writer.writeheader()
    
    for i, recipe in enumerate(data):
        # Initialize row with recipe ID
        row = {"recipe_id": i + 1}
        
        # A. Initialize ALL ingredient columns to 0.0
        for ing in all_ingredients:
            row[f"{ing}_pct"] = 0.0
        
        # B. Fill actual ingredient percentages (with cleaned names)
        for ing_name, pct in recipe["pct_ingridients"]:
            row[f"{ing_name}_pct"] = round(pct, 4)
        
        # C. Initialize ALL category columns to 0
        for category in all_keywords:
            row[category] = 0
        
        for category in recipe["processed_keywords"]:
            row[category] = 1
        
        # E. Add rating columns
        row["strength_rating"] = recipe.get("strength_rating", 0)
        row["taste_rating"] = recipe.get("taste_rating", 0)
        row["rating"] = recipe.get("rating", 0)
        
        writer.writerow(row)

print(f"CSV created successfully!")
print(f"Total recipes: {len(data)}")
print(f"Total ingredients: {len(all_ingredients)}")
print(f"Total columns: {len(header)}")
print(f"File saved: cocktail_dataset_normalized.csv")

244
10 ['bittersweet', 'citrus', 'creamy', 'floral', 'fruity', 'herbal', 'nutty', 'savoury', 'spicy', 'sweet']
CSV created successfully!
Total recipes: 2700
Total ingredients: 244
Total columns: 258
File saved: cocktail_dataset_normalized.csv


In [33]:
df = pd.read_csv("cocktail_dataset.csv")
print(f"Shape: {df.shape}")  # Should be (2700, ~258)
print(f"Missing values: {df.isnull().sum().sum()}")  # Should be 0
print(f"Ingredient columns: {[c for c in df.columns if '_pct' in c][:5]}...")

Shape: (2700, 258)
Missing values: 0
Ingredient columns: ['absinthe_pct', 'advocaat_liqueur_pct', 'agave_syrup_pct', 'aged_jamaican_rum_pct', 'aged_rum_(6-10yr)_pct']...
